# 11b — Batch Normalization and Dropout (PyTorch)

Corresponds to: Note 11 — Batch Normalization and Dropout

**Libraries introduced in this notebook:** No new libraries — `torch` and `torch.nn` introduced in Note 09.

**What this notebook demonstrates:**
1. `nn.BatchNorm1d` — forward pass verified against the 11a numpy implementation
2. Training vs inference mode — `model.train()` / `model.eval()` effect on batch norm and dropout
3. `nn.Dropout` — inverted dropout in PyTorch, verified against 11a
4. Full network with batch norm and dropout — correct layer ordering and mode switching

In [ ]:
import torch
import torch.nn as nn

print(f'PyTorch version: {torch.__version__}')

## Part 1: `nn.BatchNorm1d`

In 11a we implemented batch norm from scratch:

    μ_B  = mean(z)
    σ²_B = var(z)
    ẑ    = (z − μ_B) / √(σ²_B + ε)
    y    = γ · ẑ + β

`nn.BatchNorm1d(num_features)` does exactly this. γ is stored as `weight`, β as `bias`.

We verify it against the 11a worked example: z = [2.0, 4.0, 6.0, 8.0], γ = 2.0, β = 1.0.

In [ ]:
# nn.BatchNorm1d expects input shape (N, C) — N samples, C features
# Our worked example is 4 samples, 1 feature
z = torch.tensor([[2.0], [4.0], [6.0], [8.0]])

bn = nn.BatchNorm1d(num_features=1)

# Set γ=2.0, β=1.0 to match the worked example
with torch.no_grad():
    bn.weight.fill_(2.0)   # γ
    bn.bias.fill_(1.0)     # β

bn.train()   # training mode — uses batch statistics
with torch.no_grad():
    y = bn(z)

print('nn.BatchNorm1d output:')
print(f'  y = {y.squeeze().tolist()}')
print(f'  mean = {y.mean().item():.3f}  (expected β = 1.0)')
print(f'  std  = {y.std(unbiased=False).item():.3f}  (expected γ = 2.0)')
print()
print('11a numpy output (expected):')
print('  y = [-1.684, 0.106, 1.894, 3.684]')

The outputs match the hand calculation from Note 11 Part 2.

Note: PyTorch's `BatchNorm1d` uses Bessel's correction (unbiased variance, divides by N−1) by default
for the running variance update, but uses biased variance (divides by N) for the normalization step —
matching the formula in the note. Small numerical differences from the numpy result are expected.

## Part 2: Training vs Inference Mode

Batch norm uses batch statistics during training and running statistics during inference.
`model.train()` and `model.eval()` switch between these modes.

We demonstrate the difference by:
1. Training a batch norm layer on several batches so it accumulates running statistics
2. Switching to eval mode and showing that the output is now deterministic
3. Showing what goes wrong if eval mode is forgotten

In [ ]:
torch.manual_seed(0)
bn = nn.BatchNorm1d(num_features=4)

# Simulate training — pass several batches to accumulate running statistics
bn.train()
for _ in range(100):
    batch = torch.randn(32, 4)   # 32 samples, 4 features
    with torch.no_grad():
        bn(batch)

print('Running statistics after 100 training batches:')
print(f'  running_mean = {bn.running_mean.tolist()}')
print(f'  running_var  = {bn.running_var.tolist()}')
print()

# Single test sample
x_test = torch.tensor([[1.0, 2.0, 3.0, 4.0]])

# Eval mode — uses running statistics, deterministic
bn.eval()
with torch.no_grad():
    out_eval_1 = bn(x_test)
    out_eval_2 = bn(x_test)

print('Eval mode (model.eval()):')
print(f'  Pass 1: {out_eval_1.tolist()}')
print(f'  Pass 2: {out_eval_2.tolist()}')
print(f'  Identical: {torch.allclose(out_eval_1, out_eval_2)}')
print()

# Train mode on a single sample — batch statistics from 1 sample are meaningless
bn.train()
with torch.no_grad():
    out_train_1 = bn(x_test)
    out_train_2 = bn(x_test)

print('Train mode on a single sample (wrong — never do this at inference):')
print(f'  Pass 1: {out_train_1.tolist()}')
print(f'  Pass 2: {out_train_2.tolist()}')
print(f'  Identical: {torch.allclose(out_train_1, out_train_2)}')
print('  (outputs are always zero — mean of one value is that value, variance is zero)')

## Part 3: `nn.Dropout`

In 11a we implemented inverted dropout from scratch. `nn.Dropout(p)` does the same thing:
- Training: zero each element with probability p, scale survivors by 1/(1−p)
- Inference: pass through unchanged

We verify the training/inference behaviour and confirm that expected magnitude is preserved.

In [ ]:
torch.manual_seed(0)
dropout = nn.Dropout(p=0.5)

a = torch.tensor([0.5, 0.8, 0.3, 0.9, 0.1, 0.7, 0.4, 0.6])

# Training mode — different mask each call
dropout.train()
print('Training mode (p=0.5):')
print(f'  Input: {a.tolist()}')
for step in range(3):
    out = dropout(a)
    mask = (out != 0).int()
    print(f'  Step {step+1} mask:   {mask.tolist()}')
    print(f'  Step {step+1} output: {out.round(decimals=3).tolist()}')

print()

# Inference mode — all neurons active
dropout.eval()
out_eval = dropout(a)
print('Inference mode:')
print(f'  Output: {out_eval.tolist()}')
print(f'  Identical to input: {torch.allclose(out_eval, a)}')

In [ ]:
# Verify expected magnitude is preserved across many samples
torch.manual_seed(1)
dropout.train()
a_large = torch.randn(10000)
a_dropped = dropout(a_large)

print(f'Original mean:       {a_large.mean().item():.4f}')
print(f'Original std:        {a_large.std().item():.4f}')
print(f'After dropout mean:  {a_dropped.mean().item():.4f}  (expected ≈ original)')
print(f'After dropout std:   {a_dropped.std().item():.4f}  (expected ≈ original)')

## Part 4: Full Network with Batch Norm and Dropout

The correct pattern for a network with both batch norm and dropout:
- Batch norm placed after the linear layer, before the activation (original placement)
- Dropout placed after the activation, with a conservative rate when combined with batch norm
- Always call `model.train()` before training and `model.eval()` before evaluation

We build a 3-layer network, run a forward pass in both modes, and confirm the mode switch works.

In [ ]:
torch.manual_seed(42)

model = nn.Sequential(
    nn.Linear(16, 32),
    nn.BatchNorm1d(32),
    nn.ReLU(),
    nn.Dropout(p=0.2),

    nn.Linear(32, 16),
    nn.BatchNorm1d(16),
    nn.ReLU(),
    nn.Dropout(p=0.2),

    nn.Linear(16, 1)
)

# Apply He init to all Linear layers
for m in model.modules():
    if isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
        nn.init.zeros_(m.bias)

print('Model architecture:')
print(model)
print()
print(f'Total parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
x = torch.randn(8, 16)   # batch of 8 samples

# Training mode — batch norm uses batch stats, dropout active
model.train()
with torch.no_grad():
    out_train_1 = model(x)
    out_train_2 = model(x)

print('Training mode:')
print(f'  Pass 1 output (first 4): {out_train_1[:4].squeeze().tolist()}')
print(f'  Pass 2 output (first 4): {out_train_2[:4].squeeze().tolist()}')
print(f'  Identical: {torch.allclose(out_train_1, out_train_2)}  (expected False — dropout is random)')
print()

# Eval mode — batch norm uses running stats, dropout disabled
model.eval()
with torch.no_grad():
    out_eval_1 = model(x)
    out_eval_2 = model(x)

print('Eval mode:')
print(f'  Pass 1 output (first 4): {out_eval_1[:4].squeeze().tolist()}')
print(f'  Pass 2 output (first 4): {out_eval_2[:4].squeeze().tolist()}')
print(f'  Identical: {torch.allclose(out_eval_1, out_eval_2)}  (expected True — deterministic)')

## Summary

This notebook demonstrated:

1. **`nn.BatchNorm1d`** — produces the same output as the 11a numpy implementation. γ is stored as `weight`, β as `bias`. Output mean equals β and output std equals γ, matching the Note 11 worked example.
2. **Training vs inference mode** — `model.eval()` switches batch norm to use running statistics accumulated during training. Forgetting `model.eval()` on a single sample produces zero output (variance of one value is zero). Both batch norm and dropout are affected by the mode switch.
3. **`nn.Dropout`** — identical behaviour to the 11a hand implementation. Training mode applies a random mask and scales by 1/(1−p). Eval mode passes through unchanged. Expected magnitude is preserved across both modes.
4. **Full network** — correct ordering is Linear → BatchNorm → Activation → Dropout. Training mode is non-deterministic (dropout active), eval mode is deterministic. Always call `model.train()` before training and `model.eval()` before evaluation or inference.